In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

print("Both libraries imported successfully!")

NumPy version: 2.4.6
Pandas version: 3.0.6
Both libraries imported successfully!


In [3]:
from pathlib import Path
import warnings
import json
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"
FIGURE_DIR = REPORT_DIR / "figures"

for folder in [MODEL_DIR, REPORT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110

print("Project directory:", PROJECT_DIR.resolve())

Project directory: D:\Projects\employee-attrition-prediction


In [9]:
df = pd.read_csv("D:/Projects/employee-attrition-prediction/data/WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Create Tenure Bands
df["TenureBand"] = pd.cut(
    df["YearsAtCompany"],
    bins=[-1, 3, 10, np.inf],
    labels=["Short", "Medium", "Long"]
)

# Create a transparent workload/travel proxy.
# This is NOT a validated work-life balance measure.
overtime_score = df["OverTime"].map({"No": 0, "Yes": 1})

travel_score = df["BusinessTravel"].map({
    "Non-Travel": 0,
    "Travel_Rarely": 1,
    "Travel_Frequently": 2
})

df["WorkLifeRiskIndex"] = overtime_score + travel_score

# Promotion gap already exists as YearsSinceLastPromotion.
# Rename it instead of duplicating the same information.
df = df.rename(
    columns={"YearsSinceLastPromotion": "PromotionGap"}
)

display(df[[
    "YearsAtCompany",
    "TenureBand",
    "OverTime",
    "BusinessTravel",
    "WorkLifeRiskIndex",
    "PromotionGap"
]].head(10))

,YearsAtCompany,TenureBand,OverTime,BusinessTravel,WorkLifeRiskIndex,PromotionGap
0,6,Medium,Yes,Travel_Rarely,2,0
1,10,Medium,No,Travel_Frequently,2,1
2,0,Short,Yes,Travel_Rarely,2,0
3,8,Medium,Yes,Travel_Frequently,3,3
4,2,Short,No,Travel_Rarely,1,2
5,7,Medium,No,Travel_Frequently,2,3
6,1,Short,Yes,Travel_Rarely,2,0
7,1,Short,No,Travel_Rarely,1,0
8,9,Medium,No,Travel_Frequently,2,1
9,7,Medium,No,Travel_Rarely,1,7


In [11]:
# Constant columns and employee identifier
columns_to_drop = [
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"No": 0, "Yes": 1})

assert y.notna().all(), "Unexpected target labels found."

print("Feature matrix:", X.shape)
print("Target:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Feature matrix: (1470, 32)
Target: (1470,)

Target distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64


In [13]:
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'PromotionGap', 'YearsWithCurrManager', 'WorkLifeRiskIndex']

Categorical features: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'TenureBand']


C:\Users\adity\AppData\Local\Temp\ipykernel_4712\3959600475.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining attrition rate:")
print(round(y_train.mean() * 100, 2), "%")

print("Testing attrition rate:")
print(round(y_test.mean() * 100, 2), "%")

Training records: 1176
Testing records: 294

Training attrition rate:
16.16 %
Testing attrition rate:
15.99 %


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_preprocessor, numerical_features),
    ("cat", categorical_preprocessor, categorical_features)
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.
